# NOVA SN68 — scoring Boltz-2 sur GPU Kaggle

Note un lot de molécules avec Boltz-2 **exactement comme les validateurs NOVA** (même entrée, même MSA, mêmes paramètres, même graine) et sauvegarde les scores dans `/kaggle/working/`.

**Avant de lancer :**
1. Paramètres du notebook → *Accelerator* : **GPU T4 x2** (ou P100), *Internet* : **On** (compte vérifié par téléphone requis).
2. Pour laisser tourner sans garder l'onglet ouvert : **Save Version → Save & Run All (Commit)**, puis récupérer le CSV dans l'onglet *Output* de la version.

Usage de recherche uniquement : ce notebook ne soumet rien au réseau Bittensor.

In [ ]:
# Paramètres
BRANCH = "claude/eager-mayer-ot25x6"
BATCH = "inputs/kaggle_batch.csv"   # CSV avec une colonne 'name' (+ colonnes libres conservées)
TOP = None                          # None = tout le lot ; ex. 2 pour un test de durée rapide
RUN = "kaggle_batch1"
MULTI_GPU = False                   # True pour essayer les 2 T4 (plus rapide, moins éprouvé)

import time
T0 = time.time()
def step(msg):
    print(f"[{time.strftime('%H:%M:%S')}] +{(time.time() - T0) / 60:5.1f} min  {msg}", flush=True)
step("démarrage")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
step("test de l'accès Internet")
import subprocess
for url in ["https://github.com", "https://huggingface.co", "https://download.pytorch.org"]:
    code = subprocess.run(["curl", "-s", "-o", "/dev/null", "-m", "15", "-w", "%{http_code}", url],
                          capture_output=True, text=True).stdout or "aucune réponse"
    print(f"  {url}: {code}")
    if not code.isdigit() or code == "000":
        raise SystemExit("Pas d'accès Internet : activer 'Internet' dans les paramètres du notebook "
                         "(panneau de droite > Session options ; compte vérifié par téléphone requis), puis relancer.")
step("téléchargement du code (archive GitHub, sans git)")
!rm -rf /kaggle/working/Nova && mkdir -p /kaggle/working/Nova
!curl -sSL -m 120 --retry 3 "https://codeload.github.com/0xRozier/Nova/tar.gz/refs/heads/{BRANCH}" | tar xz --strip-components=1 -C /kaggle/working/Nova
%cd /kaggle/working/Nova
!ls
step("téléchargement de la base combinatoire (~120 Mo)")
!mkdir -p data && wget -nv --timeout=60 --tries=3 -O data/molecules.sqlite "https://huggingface.co/datasets/Metanova/Mol-Rxn-DB/resolve/main/molecules.sqlite"
!ls -la data
step("étape 1 terminée")

## Environnement isolé pour Boltz 2.2.1
Boltz 2.2.1 exige `numpy<2` et Python ≤ 3.12 : on l'installe dans un venv séparé pour ne pas casser l'image Kaggle (~5 min).

In [ ]:
step("installation de uv + Python 3.12")
!pip install -q uv
!uv venv -q --python 3.12 /kaggle/working/bvenv
step("installation de torch 2.7.1 CUDA (~2,5 Go)")
!uv pip install -q --python /kaggle/working/bvenv/bin/python torch==2.7.1 --index-url https://download.pytorch.org/whl/cu126
step("installation de boltz 2.2.1")
!uv pip install -q --python /kaggle/working/bvenv/bin/python boltz==2.2.1 "pyarrow<17" pandas requests pyyaml
!/kaggle/working/bvenv/bin/python -c "import torch, boltz; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count(), torch.cuda.get_device_name(0))"
step("étape 2 terminée")

## Précision de calcul
Les validateurs tournent en `bf16-mixed` sur des GPU récents (A100…). Les T4/P100 de Kaggle (architecture < 8.0) ne gèrent pas le bf16 nativement : dans ce cas on passe Boltz en précision 32 bits. C'est un léger écart, mesuré par les molécules de calibration du lot.

In [ ]:
import subprocess, re, pathlib
py = "/kaggle/working/bvenv/bin/python"
cap = subprocess.check_output([py, "-c", "import torch; print('%d.%d' % torch.cuda.get_device_capability(0))"], text=True).strip()
n_gpus = int(subprocess.check_output([py, "-c", "import torch; print(torch.cuda.device_count())"], text=True))
main_py = pathlib.Path(subprocess.check_output([py, "-c", "import boltz.main as m; print(m.__file__)"], text=True).strip())
print("compute capability", cap, "| GPUs", n_gpus)
if float(cap) < 8.0:
    src = main_py.read_text()
    if '"bf16-mixed"' in src:
        main_py.write_text(src.replace('"bf16-mixed"', '32'))
    print("bf16 non supporté -> précision 32 bits")
else:
    print("bf16 supporté -> réglage identique aux validateurs")
EXTRA = f"--no_kernels --devices {n_gpus if MULTI_GPU else 1}"
step(f"réglages Boltz : {EXTRA}")

## Scoring
Le premier lancement télécharge les poids Boltz-2 (~6 Go, quelques minutes). La progression s'affiche chaque fois qu'une molécule est terminée (avec une estimation du temps restant) ; toutes les 10 minutes sans nouvelle molécule, la dernière ligne du journal Boltz est affichée. Le journal complet est dans `/kaggle/working/boltz_<RUN>.log`.

Si la session s'interrompt, relancer avec le même `RUN` reprend là où elle s'était arrêtée.

In [ ]:
import pandas as pd
n_total = len(pd.read_csv(BATCH).head(TOP) if TOP else pd.read_csv(BATCH))
cmd = [py, "-m", "scripts.boltz_score", "--file", BATCH, "--accelerator", "gpu", "--run", RUN,
       "--boltz", "/kaggle/working/bvenv/bin/boltz", f"--extra={EXTRA}"] + (["--top", str(TOP)] if TOP else [])
log_path = f"/kaggle/working/boltz_{RUN}.log"
pred_dir = pathlib.Path(f"data/boltz/{RUN}/out/boltz_results_inputs/predictions")

def last_log_line():
    text = open(log_path, errors="replace").read().replace("\r", "\n")
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return lines[-1][:200] if lines else "(journal vide)"

step(f"lancement de Boltz sur {n_total} molécules")
with open(log_path, "w") as log:
    proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    done, first_done_at, last_report = 0, None, time.time()
    while proc.poll() is None:
        time.sleep(30)
        now_done = len(list(pred_dir.glob("*/affinity_*.json")))
        if now_done > done:
            if first_done_at is None:
                first_done_at = time.time()
            rate = (time.time() - first_done_at) / 60 / (now_done - 1) if now_done > 1 else None
            eta = f" | {rate:.1f} min/molécule, ~{rate * (n_total - now_done):.0f} min restantes" if rate else ""
            step(f"{now_done}/{n_total} molécules notées{eta}")
            done, last_report = now_done, time.time()
        elif time.time() - last_report > 600:
            step(f"en cours ({done}/{n_total}) — {last_log_line()}")
            last_report = time.time()

print(open(log_path, errors="replace").read().replace("\r", "\n")[-4000:])
step(f"Boltz terminé (code {proc.returncode})")
assert proc.returncode == 0, "Boltz a échoué : voir le journal ci-dessus"

In [ ]:
import shutil, pandas as pd
out = f"/kaggle/working/scores_{RUN}.csv"
shutil.copy(f"data/boltz/{RUN}/scores.csv", out)
df = pd.read_csv(out)
print(out, len(df), "molécules")
df.head(60)